## Simulating two vessels passing a lock
In this notebook, we simulate a lock on a network which two opposingly directed vessels have to pass. We add a pre-coded lock-complex object on the graph: only one vessel can be levelled at a time. The second vessel waits until the first vessel has passed the lock. 

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable

# package(s) needed for inspecting the output
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [3]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph

In [4]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad, Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad, Point(5000,0)))

# add edges
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)

# add graph to environment
env.graph = graph

In [5]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure

In [6]:
# Two ways of making the same lock chamber: 
# (1) manual chamber dimensions and distances to nodes of edge, 
# (2) a geometry that overlaps the graph with coordinates in meters: chamber dimensions and distances to nodes of edge are determined automatically
selection = 'geometric'

if selection == 'manual':
    lock_chamber = IsLockChamber(env=env,
                                 lock_length = 400,
                                 lock_width = 50,
                                 lock_depth = 10,
                                 name='Lock',      
                                 edge = ('0','1'),
                                 distance_from_start_node_to_lock_gate_A = 4800.,
                                 distance_from_end_node_to_lock_gate_B = 4800.)

if selection == 'geometric':
    lock_chamber = IsLockChamber(env=env,
                                 lock_depth = 10,
                                 name='Lock',
                                 gate_open = '1',
                                 edge = ('0','1'),
                                 geometry = Polygon([Point(-200, -25),Point(-200, 25),Point(200, 25),Point(200, -25)]))

In [7]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   distance_from_edge_start = 0)

In [8]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents

In [9]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
    {}
)

In [10]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [11]:
# create vessels from dict 
data_vessel_1 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 1",                                  # required by Identifiable
    "geometry": env.graph.nodes['0']['geometry'],        # required by Locatable
    "route": nx.dijkstra_path(env.graph, "0", "1"),      # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
}  
vessel_1 = Vessel(**data_vessel_1)
vessel_1.name = 'Vessel 1'

data_vessel_2 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 2",                                  # required by Identifiable
    "geometry": env.graph.nodes['0']['geometry'],        # required by Locatable
    "route": nx.dijkstra_path(env.graph, "0", "1"),      # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 5,                                              # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:45:00')  # required by PassesLockComplex
}  
vessel_2 = Vessel(**data_vessel_2)
vessel_2.name = 'Vessel 2'

# start the simulation
env.process(mission(env, vessel_1))
env.process(mission(env, vessel_2));

#### 3. Run simulation

In [12]:
env.run()

Vessel 1 True
Vessel 2 True


#### 4. Inspect output

In [13]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_1.logbook)

print("'{}' logbook data:".format(vessel_1.name))  
print('')

display(df)

'Vessel 1' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 0 to node 1 start,2025-01-01 00:00:00.000000,0,POINT (-0.0449157642059761 0)
1,Waiting for lock operation start,2025-01-01 00:00:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
2,Waiting for lock operation stop,2025-01-01 00:10:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
3,Sailing to first lock gate start,2025-01-01 00:10:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
4,Sailing to first lock gate stop,2025-01-01 00:30:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
5,Sailing to position in lock start,2025-01-01 00:30:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
6,Sailing to position in lock stop,2025-01-01 00:35:40.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
7,Levelling start,2025-01-01 00:40:40.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
8,Levelling stop,2025-01-01 00:50:40.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
9,Sailing to second lock gate start,2025-01-01 00:55:40.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)


In [14]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_2.logbook)

print("'{}' logbook data:".format(vessel_2.name))  
print('')

display(df)

'Vessel 2' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 0 to node 1 start,2025-01-01 00:45:00.000000,0,POINT (-0.0449157642059761 0)
1,Waiting for lock operation start,2025-01-01 00:45:00.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
2,Waiting for lock operation stop,2025-01-01 01:08:33.768898,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
3,Sailing to first lock gate start,2025-01-01 01:08:33.768898,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.0449157642059761 0)
4,Sailing to first lock gate stop,2025-01-01 01:28:33.768898,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
5,Sailing to position in lock start,2025-01-01 01:28:33.768898,"{'origin': '', 'destination': '', 'route': [],...",POINT (-0.001796630568239 0)
6,Sailing to position in lock stop,2025-01-01 01:34:13.941684,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
7,Levelling start,2025-01-01 01:39:13.941685,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
8,Levelling stop,2025-01-01 01:49:13.941685,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)
9,Sailing to second lock gate start,2025-01-01 01:54:13.941685,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0013474729261793 0)


In [15]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber.logbook)

print("'{}' logbook data:".format(lock_chamber.name))  
print('')

display(lock_df)

'Lock' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-01 00:00:00.000000,{},1
1,Lock gate closing stop,2025-01-01 00:05:00.000000,{},1
2,Lock chamber converting start,2025-01-01 00:05:00.000000,{},1
3,Lock chamber converting stop,2025-01-01 00:15:00.000000,{},0
4,Lock gate opening start,2025-01-01 00:15:00.000000,{},0
5,Lock gate opening stop,2025-01-01 00:20:00.000000,{},0
6,Lock gate closing start,2025-01-01 00:35:40.172786,{},0
7,Lock gate closing stop,2025-01-01 00:40:40.172786,{},0
8,Lock chamber converting start,2025-01-01 00:40:40.172786,{},0
9,Lock chamber converting stop,2025-01-01 00:50:40.172786,{},1


In [16]:
# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -5050, 
                        xlimmax = 5050, 
                        method='Plotly')
fig.show()

In [17]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([vessel_1, vessel_2, lock_chamber])
generate_vessel_gantt_chart(df_eventtable)